<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Cuaderno, Formulario y Álbum de Fotos: los Tres Tipos de Datos 📒
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 00
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/00%20-%20Fundamentos%20de%20Big%20Data/Para%20Dummies/02_Tipos_de_Datos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 del Módulo 00 (Tipos de datos)](../02_Tipos_de_Datos_Estructurados_No_Estructurados_Semiestructurados.ipynb). No todos los datos «se ven» igual. Piensa en tres cosas que seguro has tenido en las manos:

| Objeto | Tipo de dato | Qué tiene de particular |
|---|---|---|
| **El cuaderno cuadriculado** del tendero: cada renglón una venta, cada columna siempre lo mismo | **Estructurado** | Todo **cabe en casillas fijas** |
| **Los formularios de pedidos**: unos traen «nota», otros «dirección», otros «piso» | **Semiestructurado** | Cada uno **dice qué campos trae**, y pueden ser **distintos** |
| **La caja de fotos y los audios** del celular | **No estructurado** | **No hay casillas**: hay que *mirar* o *leer* para entender |

Al terminar podrás explicar:
1. Cómo se ve y se **consulta** una tabla (la forma más ordenada).
2. Cómo se **convierten en tabla** los formularios que cambian, y por qué quedan **huecos**.
3. Cómo una computadora «ve» un **texto** y una **foto** (¡como números!).

> 🔗 Si luego quieres ver las versiones **desde cero** (SQL, JSON, XML, *logs*, *schema-on-write* y los formatos CSV frente a Parquet con medidas reales): [cuaderno estándar](../02_Tipos_de_Datos_Estructurados_No_Estructurados_Semiestructurados.ipynb). Antes de esto: [de los tomates a la salsa](01_Importancia_Big_Data_Dummies.ipynb).

---
## 1. El Cuaderno Cuadriculado: Datos Estructurados 📒

El tendero anota cada venta con **las mismas columnas**: fecha, producto, kilos y precio por kilo. Como todo es ordenado, se pueden hacer **preguntas** muy rápido: ¿cuánto vendí de cada producto? ¿cuánto dinero entró? (Las bases de datos de los bancos y de las tiendas funcionan así; se consultan con un lenguaje llamado **SQL**.)

Las ventas de abajo son **inventadas**.

In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ventas = pd.DataFrame({
    "fecha": ["2025-03-01"] * 3 + ["2025-03-02"] * 3,
    "producto": ["papa", "cebolla", "arveja"] * 2,
    "kilos": [50, 20, 15, 60, 25, 10],
    "precio_kg": [2800, 3100, 5200] * 2,                 # pesos por kilo (inventados)
})
ventas["total"] = ventas.kilos * ventas.precio_kg
print(ventas.to_string(index=False))
print("\nTipo de cada columna:", dict(ventas.dtypes.astype(str)))

por_producto = ventas.groupby("producto")["total"].sum().sort_values(ascending=False)
print("\nDinero por producto:"); print(por_producto.to_string())
assert por_producto["papa"] == (50 + 60) * 2800 and por_producto.index[0] == "papa"
assert ventas.shape == (6, 5)                            # 6 ventas, 5 columnas: TODAS las filas tienen TODAS las columnas

---
**Lección:** en una tabla **cada renglón tiene las mismas columnas** y cada columna tiene **un tipo** (fecha, texto, número). Gracias a eso, una pregunta como *«¿cuánto dinero dejó cada producto?»* se contesta en una línea (la **papa** dejó 308 000 pesos). Esa **rigidez** es una ventaja (nadie puede anotar un precio con letras) y también una molestia: **si mañana quieres anotar algo nuevo** (por ejemplo, «lo vendí a domicilio»), tienes que **cambiar la tabla entera**.

---

## 2. Los Formularios que Cambian: Datos Semiestructurados 📝

Don Álvaro recibe **pedidos** por el celular. Cada pedido llega como un **formulario** donde **cada campo lleva su nombre** (cliente, producto, kilos...), pero **no todos traen lo mismo**: unos incluyen una **nota**, otros una **dirección** de domicilio (con barrio y piso). A este formato (que en las computadoras se llama **JSON**) se le dice **semiestructurado**: tiene orden (los campos tienen nombre) pero **flexible**.

Para analizarlos, se **aplanan** en una tabla. Lo que no vino en un pedido queda **vacío**.

In [ ]:
pedidos = [
    {"id": 1, "cliente": "Doña Rosa", "producto": "papa", "kilos": 10},
    {"id": 2, "cliente": "Don Pedro", "producto": "cebolla", "kilos": 5, "nota": "sin tierra, por favor"},
    {"id": 3, "cliente": "Lucía", "producto": "arveja", "kilos": 3, "domicilio": {"barrio": "El Centro", "piso": 2}},
    {"id": 4, "cliente": "Andrés", "producto": "papa", "kilos": 8, "nota": "llevar temprano", "domicilio": {"barrio": "El Centro"}},
]
tabla = pd.json_normalize(pedidos)                          # aplana: 'domicilio' -> 'domicilio.barrio' y 'domicilio.piso'
print(tabla.to_string(index=False))
huecos = int(tabla.isna().sum().sum())
print(f"\nLa tabla tiene {tabla.shape[0]} filas, {tabla.shape[1]} columnas y {huecos} huecos (casillas vacias).")
assert tabla.shape == (4, 7) and huecos == 7
assert tabla.loc[tabla.cliente == "Lucía", "domicilio.piso"].iloc[0] == 2

fig, ax = plt.subplots(figsize=(7.5, 2.8))
vacio = tabla.drop(columns=["id"]).isna().to_numpy()
ax.imshow(vacio, cmap="Reds", vmin=0, vmax=1.6, aspect="auto")
ax.set_xticks(range(vacio.shape[1])); ax.set_xticklabels(tabla.drop(columns=["id"]).columns, rotation=20, fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels(tabla.cliente, fontsize=8); ax.grid(False)
ax.set_title("En rojo: lo que ese formulario NO trajo (no es un error)", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

# Tres pedidos mas (los usaremos en la practica del final)
pedidos_nuevos = [
    {"id": 10, "producto": "papa", "kilos": 12, "nota": "sin prisa"},
    {"id": 11, "producto": "arveja", "kilos": 4, "descuento": 0.1},
    {"id": 12, "producto": "cebolla", "kilos": 6, "nota": "llamar antes"},
]

---
**Lección:** al aplanar, las **7 casillas vacías** no son errores: Doña Rosa **no tenía nota que escribir** y Don Pedro **no pidió domicilio**; simplemente **ese campo no aplica** a su pedido. Un cuidado importante: **no hay que «rellenar» esos huecos inventando datos** (por ejemplo, ponerle piso 1 a quien no pidió domicilio). La ventaja de los formularios flexibles es que **mañana puedes añadir un campo nuevo** sin romper nada; el costo es que **al analizar tienes que decidir qué hacer con lo que falta**.

---

## 3. La Caja de Fotos: Datos No Estructurados 📷

Un **texto** libre (una opinión de un cliente) o una **foto** no tienen casillas. La computadora, sin embargo, también los convierte en algo con lo que puede trabajar:

- Un **texto** se parte en **palabras** y se **cuentan** (las más repetidas dan una pista de qué opinan).
- Una **foto** es una **cuadrícula de números** (cada cuadrito, o *píxel*, tiene un número de brillo): aquí, 0 = oscuro y 1 = claro.

In [ ]:
opiniones = ["La papa llegó fresca y el precio fue justo",
             "El empaque venía roto, pero la papa estaba fresca",
             "Buen precio y buena atención del vendedor",
             "Llegó tarde y la arveja estaba húmeda"]
vacias = {"la", "el", "y", "del", "de", "pero", "venía", "estaba", "fue", "llegó"}                  # palabras sin mucha informacion
palabras = [p for o in opiniones for p in re.findall(r"[a-záéíóúñ]+", o.lower()) if p not in vacias]
conteo = Counter(palabras)
print("Palabras mas repetidas:", conteo.most_common(4))
assert conteo["papa"] == 2 and conteo["fresca"] == 2 and conteo["precio"] == 2

# Una foto de 5x5 pixeles: una carita (1 = claro, 0 = oscuro)
carita = np.array([[0, 1, 0, 1, 0],
                   [0, 1, 0, 1, 0],
                   [0, 0, 0, 0, 0],
                   [1, 0, 0, 0, 1],
                   [0, 1, 1, 1, 0]])
print("\nLo que 've' una persona:")
print("\n".join("".join("█" if v else "·" for v in fila) for fila in carita))
print("\nLo que 've' la computadora:", carita.tolist())
assert carita.shape == (5, 5) and carita.sum() == 9
assert carita.size * 4 > 5 * 8                              # 25 numeros ya pesan mas que una fila de 5 numeros: las fotos reales (millones de pixeles) pesan MUCHO

---
**Lección:** el texto y la imagen **no traen casillas**, pero se pueden **convertir en números**: contar palabras (en las opiniones, «papa», «fresca» y «precio» se repiten **2 veces cada una**) y mirar la imagen como una **tabla de brillo**. Una foto real tiene **millones** de cuadritos, por eso las fotos, los audios y los videos son lo que **más espacio ocupa** en el Big Data. Y fíjate: el valor de una foto **no está en un cuadrito**, sino en **cómo se relacionan** (dos ojos, una boca); por eso se necesitan programas especiales para «entenderlas».

---

## Resumen en una frase 🎯

> **Hay datos ordenados en casillas (tablas), datos con nombre pero flexibles (formularios) y datos sin casillas (texto, fotos); cada tipo se guarda y se consulta de forma distinta, y a todos se les puede sacar valor si se convierten en algo medible.**

### Lo que aprendiste hoy:
- ✅ **Estructurados (tabla):** mismas columnas siempre; se consultan rápido (la papa dejó **308 000 pesos**), pero cambiar la forma cuesta.
- ✅ **Semiestructurados (formularios/JSON):** cada registro dice qué trae; al aplanar quedan **huecos que no son errores** (aquí, 7).
- ✅ **No estructurados (texto, fotos):** hay que **convertirlos en números** (palabras contadas, píxeles); ocupan mucho espacio y su valor está en **las relaciones**.
- ✅ **¿Dónde guardar qué?** Una **tabla** ordenada y exacta va en una base de datos; los **formularios cambiantes** en un almacén flexible; las **fotos y audios** como archivos con una etiqueta que diga qué son. Y una regla de oro: **guarda el original sin tocar y publica una versión limpia**.
- ✅ **Una analogía de formatos:** guardar una tabla **por filas** es como un libro donde cada página es un cliente; guardarla **por columnas** es como un archivador con un cajón por tema: si solo quieres **las fechas**, abres **un solo cajón** (por eso los formatos por columnas, como Parquet, leen más rápido cuando necesitas pocas columnas).

> 🎓 **Siguiente paso:** [el cuaderno estándar](../02_Tipos_de_Datos_Estructurados_No_Estructurados_Semiestructurados.ipynb) consulta una base real con SQL, aplana JSON y XML, procesa *logs* y **mide** cuánto ocupa y cuánto tarda cada formato. Luego: [el viaje de un dato](03_Ciclo_de_Vida_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Cuenta los huecos de unos pedidos nuevos

Llegaron tres pedidos nuevos en la lista `pedidos_nuevos` (la definimos al final de la sección 2: cada uno es un diccionario y no todos traen los mismos campos). Con `pd.json_normalize`, igual que hicimos arriba:

1. Convierte la lista en una tabla llamada `tabla_nueva`.
2. Cuenta cuántas casillas vacías tiene en total (`huecos_nuevos`) y cuál es la **columna con más huecos** (`col_mas_vacia`).
3. Comprueba con `assert` que `tabla_nueva.shape == (3, 5)`, que `huecos_nuevos == 3` y que `col_mas_vacia == "descuento"`.

In [ ]:
# Escribe tu código aquí

# tabla_nueva = pd.json_normalize(pedidos_nuevos)
# huecos_nuevos = ...
# huecos_nuevos = ...
# col_mas_vacia = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
tabla_nueva = pd.json_normalize(pedidos_nuevos)
huecos_nuevos = int(tabla_nueva.isna().sum().sum())
col_mas_vacia = tabla_nueva.isna().sum().idxmax()
print(tabla_nueva.to_string(index=False))
print(f"Huecos: {huecos_nuevos} | columna con mas huecos: {col_mas_vacia}")

assert tabla_nueva.shape == (3, 5)
assert huecos_nuevos == 3 and col_mas_vacia == "descuento"
print("'descuento' y 'nota' solo aparecen en algunos pedidos: esos vacios significan 'no aplica', no 'dato perdido'.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué una **tabla** es más fácil de consultar que una **caja de fotos**? ¿Y por qué, aun así, las fotos pueden tener **más valor** para un negocio (por ejemplo, para un agricultor que fotografía sus cultivos)?
2. En el cuaderno de pedidos, ¿por qué **no** conviene rellenar los huecos con un valor inventado? Piensa en qué pasaría si luego calculas un «piso promedio».
3. Tienes que guardar: las ventas del día, las fotos del lote de papa y los mensajes de los clientes. **¿Dónde guardarías cada cosa** y por qué? ¿Qué «etiquetas» pondrías para poder encontrarlas después?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>